## 1. Setup & Imports

In [4]:
#Install all the dependencies if needed

!pip install yfinance pandas numpy scikit-learn nltk spacy transformers datasets newsapi-python torch tqdm joblib matplotlib seaborn

  Obtaining dependency information for download from https://files.pythonhosted.org/packages/37/45/01e7455a9659528e77a414b222326d4c525796e4f571bbabcb2e0ff3d1f4/download-0.3.5-py3-none-any.whl.metadata


ERROR: Could not find a version that satisfies the requirement en_core_web_sm (from versions: none)
ERROR: No matching distribution found for en_core_web_sm


In [7]:
!python -m spacy download en_core_web_sm #loading english language pack

     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
     --------------------------------------- 0.0/12.8 MB 325.1 kB/s eta 0:00:40
     --------------------------------------- 0.1/12.8 MB 657.6 kB/s eta 0:00:20
      -------------------------------------- 0.2/12.8 MB 952.6 kB/s eta 0:00:14
      --------------------------------------- 0.3/12.8 MB 1.3 MB/s eta 0:00:10
     - -------------------------------------- 0.5/12.8 MB 1.9 MB/s eta 0:00:07
     -- ------------------------------------- 0.7/12.8 MB 2.3 MB/s eta 0:00:06
     -- ------------------------------------- 0.9/12.8 MB 2.5 MB/s eta 0:00:05
     --- ------------------------------------ 1.0/12.8 MB 2.6 MB/s eta 0:00:05
     --- ------------------------------------ 1.2/12.8 MB 2.8 MB/s eta 0:00:05
     ---- ----------------------------------- 1.5/12.8 MB 3.0 MB/s eta

In [8]:
import pandas as pd
import numpy as np
import yfinance as yf
from datetime import datetime
import nltk
nltk.download('stopwords')
nltk.download('punkt')
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
import spacy
nlp = spacy.load('en_core_web_sm')
import re
import warnings
warnings.filterwarnings('ignore')
print('Setup complete.')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Meet\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Meet\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


Setup complete.


## 2. Download Market Data

In [31]:
TICKER = 'MSFT'
START_DATE = '2001-01-01' #To get last 7 years data on MSFT stock
END_DATE = datetime.today().strftime('%Y-%m-%d')

market = yf.download(TICKER, start=START_DATE, end=END_DATE, auto_adjust=True, progress=False)
market.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 6265 entries, 2001-01-02 to 2025-11-28
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   (Close, MSFT)   6265 non-null   float64
 1   (High, MSFT)    6265 non-null   float64
 2   (Low, MSFT)     6265 non-null   float64
 3   (Open, MSFT)    6265 non-null   float64
 4   (Volume, MSFT)  6265 non-null   int64  
dtypes: float64(4), int64(1)
memory usage: 293.7 KB


In [33]:
market = market.rename(columns={'Close':'Adj_Close'})
market['Return'] = market['Adj_Close'].pct_change()
market.reset_index(inplace=True)
market.head()

Price,Date,Adj_Close,High,Low,Open,Volume,Return
Ticker,,MSFT,MSFT,MSFT,MSFT,MSFT,
0,2001-01-02,13.247933,13.744254,13.095219,13.477004,82413200,NaN
1,2001-01-03,14.641456,14.927794,13.171583,13.190673,135962200,0.105188
2,2001-01-04,14.794173,15.424119,14.316942,14.603281,112397000,0.010430
3,2001-01-05,15.004156,15.233227,14.526925,14.813264,93414600,0.014194
4,2001-01-08,14.946886,15.195046,14.259673,14.946886,79817600,-0.003817


## 3. Load / Fetch News Data

In [ ]:
# Placeholder cell: user can load NewsAPI, yfinance news, or CSV
# Example:
# news_df = pd.read_csv('your_news.csv')
news_df = pd.DataFrame(columns=['date','text'])
news_df.head()

## 4. Preprocess Text

In [ ]:
from nltk.corpus import stopwords
STOPWORDS = set(stopwords.words('english'))

def clean_text(text):
    if pd.isna(text):
        return ''
    text = str(text).lower()
    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"[^a-z\s]", " ", text)
    tokens = word_tokenize(text)
    tokens = [t for t in tokens if t not in STOPWORDS and len(t) > 2]
    doc = nlp(' '.join(tokens))
    return ' '.join([token.lemma_ for token in doc])

if not news_df.empty:
    news_df['clean_text'] = news_df['text'].apply(clean_text)

news_df.head()

## 5. Generate Sentiment Labels (FinBERT or VADER)

In [ ]:
# Placeholder for FinBERT or VADER
from nltk.sentiment.vader import SentimentIntensityAnalyzer
nltk.download('vader_lexicon')
sia = SentimentIntensityAnalyzer()

news_df['sentiment_label'] = news_df['clean_text'].apply(lambda t: 1 if sia.polarity_scores(t)['compound']>0.05 else (-1 if sia.polarity_scores(t)['compound']<-0.05 else 0))
news_df.head()

## 6. Aggregate Daily Sentiment

In [ ]:
daily = news_df.groupby('date')['sentiment_label'].agg(['count','mean']).reset_index()
daily = daily.rename(columns={'count':'headlines_count','mean':'mean_sent'})
daily.head()

## 7. Merge with Market Data

In [ ]:
market['Date_only'] = market['Date'].dt.date
merged = market.merge(daily, left_on='Date_only', right_on='date', how='left')
merged['mean_sent'] = merged['mean_sent'].fillna(0)
merged['headlines_count'] = merged['headlines_count'].fillna(0)
merged['Return_next'] = merged['Return'].shift(-1)
merged.dropna(subset=['Return_next'], inplace=True)
merged.head()

## 8. Modelling – Linear Regression & Random Forest

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

features = ['mean_sent','headlines_count','High','Low','Open']
merged['volatility'] = (merged['High'] - merged['Low'])/merged['Open']
features.append('volatility')

X = merged[features].fillna(0)
y = merged['Return_next']

split = int(len(X)*0.8)
X_train, X_test = X.iloc[:split], X.iloc[split:]
y_train, y_test = y.iloc[:split], y.iloc[split:]

lr = LinearRegression().fit(X_train, y_train)
rf = RandomForestRegressor(n_estimators=100, random_state=42).fit(X_train, y_train)

y_pred_lr = lr.predict(X_test)
y_pred_rf = rf.predict(X_test)

print('LR RMSE:', mean_squared_error(y_test, y_pred_lr, squared=False))
print('RF RMSE:', mean_squared_error(y_test, y_pred_rf, squared=False))

## 9. Plot Sentiment & Returns

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10,4))
plt.plot(merged['Date'], merged['mean_sent'].rolling(7).mean())
plt.title('7-day Rolling Sentiment')
plt.show()

## 10. Save Outputs

In [ ]:
merged.to_csv('merged_market_sentiment.csv', index=False)
print('Saved merged dataset.')